# 05 — FR Error Diagnostic : Qu'est-ce qui fait exploser le RMSE ?

**Objectif** : Identifier precisement QUAND et POURQUOI le modele se trompe on `fr_spot`.

Le RMSE actuel est ~26-27 EUR. Le prix moyen FR en 2024 est ~47 EUR → 55% d'erreur relative.
Le RMSE est domine par les **grosses erreurs au carre**. Trouvons-les.

In [ ]:
import sys, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (16, 5)
plt.rcParams['font.size'] = 11

sys.path.insert(0, '..')
from src.data_loading import load_data, merge_train
from src.feature_engineering import build_features
from catboost import CatBoostRegressor, Pool
import yaml

with open('../config.yaml') as f:
    config = yaml.safe_load(f)

x_train, y_train, x_test = load_data('../data/raw')
train = merge_train(x_train, y_train)
train_fe = build_features(train, config)

holdout_start = config['validation']['holdout_start']
mask_val = train_fe['datetime_CET'] >= holdout_start
df_train = train_fe[~mask_val].copy()
df_val = train_fe[mask_val].copy()

with open('../outputs/shap_ranking_v3_clean.json') as f:
    clean_ranking = json.load(f)

print(f'Train: {df_train.shape}, Val: {df_val.shape}')
print(f'Val period: {df_val["datetime_CET"].min()} -> {df_val["datetime_CET"].max()}')

## 1. Train le modele et genere les residus

In [ ]:
target = 'fr_spot'
top20 = [f for f in clean_ranking[target][:20] if f in df_train.columns]

X_tr = df_train[top20]
X_va = df_val[top20]
y_tr = df_train[target]
y_va = df_val[target]

# Train avec arcsinh target (best config)
params = {
    'loss_function': 'RMSE', 'eval_metric': 'RMSE',
    'iterations': 5000, 'learning_rate': 0.03, 'depth': 8,
    'l2_leaf_reg': 5, 'subsample': 0.8, 'random_seed': 42,
    'verbose': 0, 'allow_writing_files': False, 'use_best_model': True,
}

# Model A: raw target
model_raw = CatBoostRegressor(**params)
model_raw.fit(Pool(X_tr, y_tr), eval_set=Pool(X_va, y_va), early_stopping_rounds=100, verbose=0)
preds_raw = model_raw.predict(X_va)

# Model B: arcsinh target
model_asinh = CatBoostRegressor(**params)
model_asinh.fit(Pool(X_tr, np.arcsinh(y_tr)), eval_set=Pool(X_va, np.arcsinh(y_va)),
                early_stopping_rounds=100, verbose=0)
preds_asinh = np.sinh(model_asinh.predict(X_va))

# Build residuals dataframe
res = df_val[['datetime_CET', 'fr_spot', 'fr_spot_la', 'hour', 'month', 'day_of_week',
              'fr_load_f', 'fr_wind_f', 'fr_solar_f', 'fr_nuclear_avcap_f',
              'fr_residual_load', 'fr_thermal_need', 'fr_scarcity_ratio']].copy()
res['pred_raw'] = preds_raw
res['pred_asinh'] = preds_asinh
res['error_raw'] = res['fr_spot'] - res['pred_raw']
res['error_asinh'] = res['fr_spot'] - res['pred_asinh']
res['sq_error_raw'] = res['error_raw'] ** 2
res['sq_error_asinh'] = res['error_asinh'] ** 2
res['abs_error_raw'] = res['error_raw'].abs()
res['abs_error_asinh'] = res['error_asinh'].abs()

rmse_raw = np.sqrt(res['sq_error_raw'].mean())
rmse_asinh = np.sqrt(res['sq_error_asinh'].mean())
print(f'RMSE raw: {rmse_raw:.3f}')
print(f'RMSE asinh: {rmse_asinh:.3f}')
print(f'Val samples: {len(res)}')

## 2. Contribution au RMSE par mois

Le RMSE total est la racine de la moyenne des erreurs carrees.
Decomposons : quel mois contribue le plus ?

In [ ]:
res['year_month'] = res['datetime_CET'].dt.to_period('M')

monthly = res.groupby('year_month').agg(
    n=('sq_error_raw', 'count'),
    rmse_raw=('sq_error_raw', lambda x: np.sqrt(x.mean())),
    rmse_asinh=('sq_error_asinh', lambda x: np.sqrt(x.mean())),
    mean_price=('fr_spot', 'mean'),
    std_price=('fr_spot', 'std'),
    max_price=('fr_spot', 'max'),
    min_price=('fr_spot', 'min'),
    sum_sq_error=('sq_error_raw', 'sum'),
).round(2)

# Contribution % au RMSE total
total_sq_error = res['sq_error_raw'].sum()
monthly['pct_of_rmse'] = (monthly['sum_sq_error'] / total_sq_error * 100).round(1)

print('Contribution au RMSE par mois :')
print(monthly[['n', 'rmse_raw', 'rmse_asinh', 'mean_price', 'std_price', 'max_price', 'min_price', 'pct_of_rmse']].to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

months = [str(m) for m in monthly.index]
axes[0].bar(months, monthly['rmse_raw'], alpha=0.7, label='Raw', color='steelblue')
axes[0].bar(months, monthly['rmse_asinh'], alpha=0.7, label='Arcsinh', color='coral')
axes[0].set_title('RMSE par mois (val set)')
axes[0].set_ylabel('RMSE (EUR)')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(months, monthly['pct_of_rmse'], color='darkred', alpha=0.7)
axes[1].set_title('% contribution au RMSE total')
axes[1].set_ylabel('%')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 3. RMSE par heure de la journee

Quelles heures sont les plus difficiles a predire ?

In [ ]:
hourly = res.groupby('hour').agg(
    rmse_raw=('sq_error_raw', lambda x: np.sqrt(x.mean())),
    rmse_asinh=('sq_error_asinh', lambda x: np.sqrt(x.mean())),
    mean_error=('error_raw', 'mean'),
    mean_price=('fr_spot', 'mean'),
    std_price=('fr_spot', 'std'),
    pct_sq=('sq_error_raw', lambda x: x.sum()),
).round(2)
hourly['pct_of_rmse'] = (hourly['pct_sq'] / total_sq_error * 100).round(1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(hourly.index, hourly['rmse_raw'], alpha=0.7, color='steelblue', label='Raw')
axes[0].bar(hourly.index, hourly['rmse_asinh'], alpha=0.5, color='coral', label='Arcsinh')
axes[0].set_title('RMSE par heure')
axes[0].set_xlabel('Heure CET')
axes[0].set_ylabel('RMSE (EUR)')
axes[0].legend()

axes[1].bar(hourly.index, hourly['mean_error'], color='darkgreen', alpha=0.7)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_title('Biais moyen par heure (actual - pred)')
axes[1].set_xlabel('Heure CET')
axes[1].set_ylabel('Biais (EUR)')

axes[2].bar(hourly.index, hourly['pct_of_rmse'], color='darkred', alpha=0.7)
axes[2].set_title('% contribution au RMSE par heure')
axes[2].set_xlabel('Heure CET')
axes[2].set_ylabel('%')

plt.tight_layout()
plt.show()

print('\nTop-5 heures les plus difficiles :')
print(hourly.sort_values('rmse_raw', ascending=False)[['rmse_raw', 'rmse_asinh', 'mean_error', 'mean_price', 'std_price', 'pct_of_rmse']].head())

## 4. Les PIRES erreurs — Top-50 heures qui detruisent le RMSE

Le RMSE est domine par les erreurs au carre. Voyons les 50 pires heures.

In [ ]:
# Top-50 worst errors
worst50 = res.nlargest(50, 'sq_error_raw')[[
    'datetime_CET', 'fr_spot', 'pred_raw', 'error_raw', 'abs_error_raw',
    'hour', 'fr_load_f', 'fr_nuclear_avcap_f', 'fr_wind_f', 'fr_scarcity_ratio'
]].copy()

# Contribution des 50 pires au RMSE total
top50_contrib = res.nlargest(50, 'sq_error_raw')['sq_error_raw'].sum() / total_sq_error * 100
top20_contrib = res.nlargest(20, 'sq_error_raw')['sq_error_raw'].sum() / total_sq_error * 100
top100_contrib = res.nlargest(100, 'sq_error_raw')['sq_error_raw'].sum() / total_sq_error * 100

print(f'Contribution au RMSE total :')
print(f'  Top-20 pires heures : {top20_contrib:.1f}%')
print(f'  Top-50 pires heures : {top50_contrib:.1f}%')
print(f'  Top-100 pires heures : {top100_contrib:.1f}%')
print(f'  (sur {len(res)} heures au total)\n')

print('Top-30 pires erreurs :')
print(worst50.head(30).to_string(index=False))

In [ ]:
# Scatter : quand sont ces grosses erreurs ?
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# Prix reel vs predit
axes[0].plot(res['datetime_CET'], res['fr_spot'], alpha=0.5, linewidth=0.5, label='Actual', color='black')
axes[0].plot(res['datetime_CET'], res['pred_raw'], alpha=0.5, linewidth=0.5, label='Predicted', color='steelblue')
axes[0].set_title('FR Spot : Actual vs Predicted (val set)')
axes[0].set_ylabel('EUR/MWh')
axes[0].legend()

# Erreurs
colors = np.where(res['abs_error_raw'] > 50, 'red', np.where(res['abs_error_raw'] > 25, 'orange', 'gray'))
axes[1].scatter(res['datetime_CET'], res['error_raw'], c=colors, s=2, alpha=0.5)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].axhline(50, color='red', linewidth=0.5, linestyle='--', label='>50 EUR error')
axes[1].axhline(-50, color='red', linewidth=0.5, linestyle='--')
axes[1].set_title('Erreurs de prediction (rouge = |erreur| > 50 EUR)')
axes[1].set_ylabel('Erreur (EUR)')
axes[1].set_xlabel('Date')
axes[1].legend()

axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout()
plt.show()

# Stats sur les grosses erreurs
n_big = (res['abs_error_raw'] > 50).sum()
n_huge = (res['abs_error_raw'] > 100).sum()
n_extreme = (res['abs_error_raw'] > 200).sum()
print(f'Erreurs > 50 EUR : {n_big} heures ({n_big/len(res)*100:.1f}%)')
print(f'Erreurs > 100 EUR : {n_huge} heures ({n_huge/len(res)*100:.1f}%)')
print(f'Erreurs > 200 EUR : {n_extreme} heures ({n_extreme/len(res)*100:.1f}%)')

## 5. Decomposition par REGIME

Separons les erreurs par type de regime de marche.

In [ ]:
# Categorisation des regimes
res['regime'] = 'Normal'
res.loc[res['fr_spot'] < 0, 'regime'] = 'Negatif'
res.loc[res['fr_spot'] > 100, 'regime'] = 'Spike >100'
res.loc[res['fr_spot'] > 200, 'regime'] = 'Spike >200'
res.loc[res['fr_spot'] < -50, 'regime'] = 'Tres negatif'

regime_stats = res.groupby('regime').agg(
    n=('sq_error_raw', 'count'),
    rmse_raw=('sq_error_raw', lambda x: np.sqrt(x.mean())),
    rmse_asinh=('sq_error_asinh', lambda x: np.sqrt(x.mean())),
    mean_price=('fr_spot', 'mean'),
    mean_error=('error_raw', 'mean'),
    sum_sq=('sq_error_raw', 'sum'),
).round(2)
regime_stats['pct_hours'] = (regime_stats['n'] / len(res) * 100).round(1)
regime_stats['pct_rmse'] = (regime_stats['sum_sq'] / total_sq_error * 100).round(1)

print('RMSE par regime :')
print(regime_stats[['n', 'pct_hours', 'rmse_raw', 'rmse_asinh', 'mean_price', 'mean_error', 'pct_rmse']].to_string())

fig, ax = plt.subplots(figsize=(10, 5))
regimes = regime_stats.index.tolist()
x = range(len(regimes))
width = 0.35
ax.bar([i-width/2 for i in x], regime_stats['pct_hours'], width, label='% des heures', color='steelblue', alpha=0.7)
ax.bar([i+width/2 for i in x], regime_stats['pct_rmse'], width, label='% du RMSE', color='darkred', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(regimes)
ax.set_title('Proportion des heures vs contribution au RMSE par regime')
ax.set_ylabel('%')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Analyse des SPIKES : quand le prix depasse 100 EUR

Les spikes sont-ils previsibles ? Quels fondamentaux les accompagnent ?

In [ ]:
spikes = res[res['fr_spot'] > 100].copy()
print(f'Nombre de spikes (>100 EUR) : {len(spikes)} ({len(spikes)/len(res)*100:.1f}% des heures)')

if len(spikes) > 0:
    print(f'\nStats des spikes :')
    print(f'  Prix moyen : {spikes["fr_spot"].mean():.1f} EUR')
    print(f'  Prix max : {spikes["fr_spot"].max():.1f} EUR')
    print(f'  Erreur moyenne : {spikes["error_raw"].mean():.1f} EUR (biais : {"sous-prediction" if spikes["error_raw"].mean() > 0 else "sur-prediction"})')
    print(f'  RMSE sur spikes : {np.sqrt(spikes["sq_error_raw"].mean()):.1f} EUR')
    print(f'  Contribution au RMSE total : {spikes["sq_error_raw"].sum() / total_sq_error * 100:.1f}%')
    
    print(f'\nHeures des spikes :')
    print(spikes['hour'].value_counts().sort_index().to_string())
    
    print(f'\nDates des spikes :')
    spike_dates = spikes['datetime_CET'].dt.date.value_counts().sort_index()
    print(spike_dates.to_string())
    
    # Plot spike days in detail
    spike_days = spikes['datetime_CET'].dt.date.unique()
    print(f'\n{len(spike_days)} jours distincts avec spikes')
    
    # Show the worst spike days
    for day in spike_days[:min(5, len(spike_days))]:
        day_data = res[res['datetime_CET'].dt.date == day]
        fig, ax = plt.subplots(figsize=(14, 4))
        hours = day_data['hour']
        ax.plot(hours, day_data['fr_spot'], 'ko-', label='Actual', linewidth=2)
        ax.plot(hours, day_data['pred_raw'], 's--', label='Pred raw', color='steelblue')
        ax.plot(hours, day_data['pred_asinh'], '^--', label='Pred asinh', color='coral')
        ax.fill_between(hours, day_data['fr_spot'], day_data['pred_raw'], alpha=0.2, color='red')
        ax.set_title(f'Spike day : {day} — Max price = {day_data["fr_spot"].max():.0f} EUR')
        ax.set_xlabel('Heure CET')
        ax.set_ylabel('EUR/MWh')
        ax.legend()
        plt.tight_layout()
        plt.show()
        
        print(f'  Load: {day_data["fr_load_f"].mean():.0f} MW, Nuclear: {day_data["fr_nuclear_avcap_f"].mean():.0f} MW, Wind: {day_data["fr_wind_f"].mean():.0f} MW')
        print(f'  Scarcity ratio: {day_data["fr_scarcity_ratio"].mean():.2f}')
        print()

## 7. Analyse des prix NEGATIFS

Les prix negatifs sont-ils also un probleme for le RMSE ?

In [ ]:
negatives = res[res['fr_spot'] < 0].copy()
print(f'Heures avec prix negatif : {len(negatives)} ({len(negatives)/len(res)*100:.1f}%)')

if len(negatives) > 0:
    print(f'\nStats des prix negatifs :')
    print(f'  Prix moyen : {negatives["fr_spot"].mean():.1f} EUR')
    print(f'  Prix min : {negatives["fr_spot"].min():.1f} EUR')
    print(f'  Prediction moyenne : {negatives["pred_raw"].mean():.1f} EUR (biais : le modele predit-il du negatif ?)')
    print(f'  % des preds qui sont negatives : {(negatives["pred_raw"] < 0).mean()*100:.1f}%')
    print(f'  RMSE sur negatifs : {np.sqrt(negatives["sq_error_raw"].mean()):.1f} EUR')
    print(f'  Contribution au RMSE total : {negatives["sq_error_raw"].sum() / total_sq_error * 100:.1f}%')
    
    print(f'\nHeures des prix negatifs :')
    print(negatives['hour'].value_counts().sort_index().to_string())
    
    # Prix negatif : est-ce que le modele suit ?
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].scatter(negatives['fr_spot'], negatives['pred_raw'], alpha=0.3, s=10, color='steelblue')
    axes[0].plot([-200, 0], [-200, 0], 'r--', linewidth=1)
    axes[0].set_xlabel('Actual (EUR)')
    axes[0].set_ylabel('Predicted (EUR)')
    axes[0].set_title('Prix negatifs : Actual vs Predicted (raw)')
    
    axes[1].scatter(negatives['fr_spot'], negatives['pred_asinh'], alpha=0.3, s=10, color='coral')
    axes[1].plot([-200, 0], [-200, 0], 'r--', linewidth=1)
    axes[1].set_xlabel('Actual (EUR)')
    axes[1].set_ylabel('Predicted (EUR)')
    axes[1].set_title('Prix negatifs : Actual vs Predicted (arcsinh)')
    
    plt.tight_layout()
    plt.show()

## 8. Scatter global : Actual vs Predicted

Ou le modele echoue systematiquement ?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, pred_col, title in [(axes[0], 'pred_raw', 'Raw target'),
                             (axes[1], 'pred_asinh', 'Arcsinh target')]:
    ax.scatter(res['fr_spot'], res[pred_col], alpha=0.15, s=5, color='steelblue')
    lims = [res['fr_spot'].min() - 10, res['fr_spot'].max() + 10]
    ax.plot(lims, lims, 'r-', linewidth=1, label='Perfect')
    ax.set_xlabel('Actual FR Spot (EUR)')
    ax.set_ylabel('Predicted (EUR)')
    ax.set_title(f'{title}')
    ax.legend()

plt.suptitle('Actual vs Predicted — FR Spot', fontsize=14)
plt.tight_layout()
plt.show()

# Quantile analysis : the model compresses extremes
quantiles = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
q_actual = res['fr_spot'].quantile(quantiles)
q_pred_raw = pd.Series(res['pred_raw'].values).quantile(quantiles)
q_pred_asinh = pd.Series(res['pred_asinh'].values).quantile(quantiles)

q_df = pd.DataFrame({
    'Actual': q_actual.values,
    'Pred_raw': q_pred_raw.values,
    'Pred_asinh': q_pred_asinh.values,
}, index=[f'{q:.0%}' for q in quantiles])
q_df['Gap_raw'] = q_df['Pred_raw'] - q_df['Actual']
q_df['Gap_asinh'] = q_df['Pred_asinh'] - q_df['Actual']

print('Distribution des predictions vs actual (quantiles) :')
print(q_df.round(1).to_string())

## 9. Erreur par jour de la semaine

Weekend vs weekday ? Le modele gere-t-il la difference ?

In [ ]:
dow_names = ['Lundi', 'Mardi', 'Mercredi', 'Jeudi', 'Vendredi', 'Samedi', 'Dimanche']
dow_stats = res.groupby('day_of_week').agg(
    rmse_raw=('sq_error_raw', lambda x: np.sqrt(x.mean())),
    rmse_asinh=('sq_error_asinh', lambda x: np.sqrt(x.mean())),
    mean_error=('error_raw', 'mean'),
    mean_price=('fr_spot', 'mean'),
    pct_sq=('sq_error_raw', lambda x: x.sum()),
).round(2)
dow_stats['pct_rmse'] = (dow_stats['pct_sq'] / total_sq_error * 100).round(1)
dow_stats.index = dow_names

print('RMSE par jour de la semaine :')
print(dow_stats[['rmse_raw', 'rmse_asinh', 'mean_error', 'mean_price', 'pct_rmse']].to_string())

fig, ax = plt.subplots(figsize=(10, 5))
x = range(7)
ax.bar(x, dow_stats['rmse_raw'], alpha=0.7, color='steelblue', label='Raw')
ax.bar(x, dow_stats['rmse_asinh'], alpha=0.5, color='coral', label='Arcsinh')
ax.set_xticks(x)
ax.set_xticklabels(dow_names, rotation=45)
ax.set_title('RMSE par jour de la semaine')
ax.set_ylabel('RMSE (EUR)')
ax.legend()
plt.tight_layout()
plt.show()

## 10. Erreur vs fondamentaux : quel facteur correle le plus with les grosses erreurs ?

In [ ]:
# Correlation entre |erreur| et fondamentaux
fundamentals = [
    'fr_spot', 'fr_spot_la', 'fr_load_f', 'fr_nuclear_avcap_f', 'fr_wind_f',
    'fr_solar_f', 'fr_residual_load', 'fr_thermal_need', 'fr_scarcity_ratio',
    'hour',
]

corr_with_error = res[fundamentals + ['abs_error_raw']].corr()['abs_error_raw'].drop('abs_error_raw').sort_values(ascending=False)

print('Correlation |erreur| vs fondamentaux :')
for feat, corr in corr_with_error.items():
    print(f'  {corr:+.3f}  {feat}')

# Scatter des 3 plus correles
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
top3 = corr_with_error.abs().nlargest(3).index.tolist()

for ax, feat in zip(axes, top3):
    ax.scatter(res[feat], res['abs_error_raw'], alpha=0.15, s=5, color='steelblue')
    ax.set_xlabel(feat)
    ax.set_ylabel('|Erreur| (EUR)')
    ax.set_title(f'r = {corr_with_error[feat]:.3f}')

plt.suptitle('Quels fondamentaux correlent avec les erreurs ?', fontsize=13)
plt.tight_layout()
plt.show()

## 11. Zoom semaine-type : patterns recurrents d'erreur

Y a-t-il un pattern hebdomadaire in les erreurs ?

In [ ]:
# Rolling RMSE (7 jours)
res_sorted = res.sort_values('datetime_CET')
res_sorted['rolling_rmse_7d'] = res_sorted['sq_error_raw'].rolling(168, min_periods=24).apply(lambda x: np.sqrt(x.mean()))
res_sorted['rolling_rmse_asinh_7d'] = res_sorted['sq_error_asinh'].rolling(168, min_periods=24).apply(lambda x: np.sqrt(x.mean()))

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

axes[0].plot(res_sorted['datetime_CET'], res_sorted['fr_spot'], linewidth=0.5, alpha=0.7, color='black', label='FR Spot')
axes[0].set_title('Prix FR Spot (val set)')
axes[0].set_ylabel('EUR/MWh')
axes[0].legend()

axes[1].plot(res_sorted['datetime_CET'], res_sorted['rolling_rmse_7d'], linewidth=1.5, color='steelblue', label='RMSE 7j (raw)')
axes[1].plot(res_sorted['datetime_CET'], res_sorted['rolling_rmse_asinh_7d'], linewidth=1.5, color='coral', label='RMSE 7j (arcsinh)')
axes[1].axhline(rmse_raw, color='steelblue', linewidth=0.5, linestyle='--', alpha=0.5)
axes[1].axhline(rmse_asinh, color='coral', linewidth=0.5, linestyle='--', alpha=0.5)
axes[1].set_title('RMSE glissant 7 jours')
axes[1].set_ylabel('RMSE (EUR)')
axes[1].set_xlabel('Date')
axes[1].legend()

axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout()
plt.show()

# Periodes avec RMSE > 2x la moyenne
bad_periods = res_sorted[res_sorted['rolling_rmse_7d'] > 2 * rmse_raw][['datetime_CET', 'rolling_rmse_7d']]
if len(bad_periods) > 0:
    print(f'\nPeriodes avec RMSE > {2*rmse_raw:.0f} EUR (2x moyenne) :')
    bad_periods['date'] = bad_periods['datetime_CET'].dt.date
    bad_dates = bad_periods.groupby('date')['rolling_rmse_7d'].max()
    print(bad_dates.to_string())
else:
    print('\nAucune periode avec RMSE > 2x la moyenne')

## 12. Synthese & Recommandations

In [ ]:
print('=' * 60)
print('  SYNTHESE — FR Error Diagnostic')
print('=' * 60)

print(f'\nRMSE raw:    {rmse_raw:.2f} EUR')
print(f'RMSE arcsinh: {rmse_asinh:.2f} EUR')
print(f'Gain arcsinh: {rmse_raw - rmse_asinh:.2f} EUR ({(rmse_raw - rmse_asinh)/rmse_raw*100:.1f}%)')

print(f'\nDecomposition du RMSE :')
for regime, row in regime_stats.iterrows():
    print(f'  {regime:20s}: {row["pct_hours"]:5.1f}% des heures, {row["pct_rmse"]:5.1f}% du RMSE (RMSE={row["rmse_raw"]:.1f})')

print(f'\nConcentration des erreurs :')
print(f'  Top-20 heures : {top20_contrib:.1f}% du RMSE')
print(f'  Top-50 heures : {top50_contrib:.1f}% du RMSE')
print(f'  Top-100 heures : {top100_contrib:.1f}% du RMSE')

print(f'\nHeures les plus correlees aux erreurs :')
for feat, corr in corr_with_error.head(3).items():
    print(f'  {feat}: r={corr:.3f}')